# FILA 차집합 ABSA 추론 — 옵션 4-A

**목적**: 라벨러1(안진식)의 stratified sample(`absa_phase_e_sample.parquet`)에 포함되지 않은 FILA 리뷰 약 17,014건을 ABSA v9로 추론하여, 라벨러1 결과와 합쳐 **FILA 전수 분석**을 완성한다.

**환경**: Mac M4 Pro 로컬 + Ollama EXAONE 3.5 7.8B-Instruct-Q4_K_M

**작업 흐름**:
1. 환경 점검 (Ollama 연결, kiwipiepy 설치 확인)
2. FILA 차집합 추출 → `absa_fila_complement.parquet`
3. **100건 벤치마크** → 실측 시간 산정
4. 본 추론 (체크포인트 50건마다)
5. 결과 저장 → `absa_fila_complement_predictions.parquet`

**주의**: 셀 단위로 차례로 실행. 벤치마크 결과 확인 후 본 추론 셀을 실행할지 결정한다.

## 1. 환경 셋업

In [1]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

# 작업 폴더로 이동 (absa_v9 임포트 + 골든셋 상대경로 위해 필수)
WORK_DIR = Path('/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/absa_exaone')
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

FULL_PARQUET   = Path('/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/final_data/preprocessed_absa.parquet')
SAMPLE_PARQUET = WORK_DIR / 'absa_phase_e_sample.parquet'
GOLDEN_XLSX    = WORK_DIR / 'absa_golden_set_1000_v23.xlsx'

COMPLEMENT_PARQUET   = WORK_DIR / 'absa_fila_complement.parquet'
BENCHMARK_CSV        = WORK_DIR / 'absa_fila_benchmark_100.csv'
CHECKPOINT_CSV       = WORK_DIR / 'absa_fila_complement_checkpoint.csv'
PREDICTIONS_PARQUET  = WORK_DIR / 'absa_fila_complement_predictions.parquet'

for p in [FULL_PARQUET, SAMPLE_PARQUET, GOLDEN_XLSX]:
    assert p.exists(), f'필수 파일 누락: {p}'
print('경로 점검 OK')
print(f'  WORK_DIR        = {WORK_DIR}')
print(f'  FULL_PARQUET    = {FULL_PARQUET.name}')
print(f'  SAMPLE_PARQUET  = {SAMPLE_PARQUET.name}')
print(f'  GOLDEN_XLSX     = {GOLDEN_XLSX.name}')

경로 점검 OK
  WORK_DIR        = /Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/absa_exaone
  FULL_PARQUET    = preprocessed_absa.parquet
  SAMPLE_PARQUET  = absa_phase_e_sample.parquet
  GOLDEN_XLSX     = absa_golden_set_1000_v23.xlsx


In [2]:
# Ollama 연결 + EXAONE 모델 보유 확인
import ollama

MODEL = 'exaone3.5:7.8b'

models = [m['model'] for m in ollama.list().get('models', [])]
print(f'Ollama 보유 모델 ({len(models)}개):')
for m in models:
    print(f'  - {m}')

assert any(MODEL in m for m in models), f'{MODEL} 미설치 — `ollama pull {MODEL}` 실행 필요'

# 1회 워밍업 호출 (모델 로드 시간 측정 + 정상 응답 확인)
t0 = time.time()
resp = ollama.generate(model=MODEL, prompt='안녕하세요. 한 문장으로 답해 주세요.',
                       options={'temperature': 0.1, 'num_predict': 30})
elapsed = time.time() - t0
print(f'\n워밍업 응답 ({elapsed:.1f}s):')
print(resp['response'].strip()[:200])

Ollama 보유 모델 (1개):
  - exaone3.5:7.8b

워밍업 응답 (1.2s):
안녕하세요! 어떻게 도와드릴까요? 😊


In [3]:
# kiwipiepy + absa_v9 임포트 확인
import importlib
import absa_v9
importlib.reload(absa_v9)

from absa_v9 import (
    ASPECTS, predict_dataframe, build_few_shot_examples,
    load_golden_set, preprocess_texts,
)

print(f'absa_v9 로드 OK')
print(f'  속성: {ASPECTS}')

# KiWi 1회 워밍업 (첫 호출이 느려서 본 추론 시간에 섞이지 않게)
_ = preprocess_texts(['워밍업 테스트입니다. 부드러운 원단이 마음에 듭니다.'])
print('KiWi 워밍업 OK')

absa_v9 로드 OK
  속성: ['핏/사이즈', '소재/내구성', '기능성', '디자인', '브랜드/헤리티지', '가격/가치']
KiWi 워밍업 OK


## 2. FILA 차집합 추출 (라벨러1 sample 제외)

In [4]:
# 1) 라벨러1 sample에서 FILA review_id 추출
sample_df  = pd.read_parquet(SAMPLE_PARQUET, columns=['review_id', 'brand'])
fila_sample_ids = set(sample_df.loc[sample_df['brand'] == 'FILA', 'review_id'])
print(f'라벨러1 sample 내 FILA 건수: {len(fila_sample_ids):,}')

# 2) 전체 데이터에서 FILA 추출
full_df = pd.read_parquet(FULL_PARQUET)
fila_all = full_df[full_df['brand'] == 'FILA'].copy()
print(f'전체 FILA 건수:               {len(fila_all):,}')

# 3) 차집합 = 전체 − sample
fila_complement = fila_all[~fila_all['review_id'].isin(fila_sample_ids)].copy()
fila_complement = fila_complement.reset_index(drop=True)
print(f'차집합(송원우 처리 대상):      {len(fila_complement):,}')

# 4) 검증
assert len(fila_complement) + len(fila_sample_ids) == len(fila_all), '차집합 크기 불일치'
assert len(set(fila_complement['review_id']) & fila_sample_ids) == 0, '교집합 존재 (중복!)'
print('\n검증 OK — 라벨러1 sample 과 0건 중복')

라벨러1 sample 내 FILA 건수: 3,014
전체 FILA 건수:               20,028
차집합(송원우 처리 대상):      17,014

검증 OK — 라벨러1 sample 과 0건 중복


In [5]:
# 차집합 parquet 저장 (재실행 시 동일 sample 보장)
fila_complement.to_parquet(COMPLEMENT_PARQUET, index=False)
print(f'저장 완료: {COMPLEMENT_PARQUET}')
print(f'  행 {len(fila_complement):,} × 열 {len(fila_complement.columns)}')

# 본 추론에 필요한 핵심 컬럼만 노출
core_cols = ['review_id', 'brand', 'rating', 'content_clean']
if 'tokens' in fila_complement.columns:
    core_cols.append('tokens')
print(f'\n핵심 컬럼 미리보기:')
fila_complement[core_cols].head(3)

저장 완료: /Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/absa_exaone/absa_fila_complement.parquet
  행 17,014 × 열 35

핵심 컬럼 미리보기:


,review_id,brand,rating,content_clean,tokens
0,143752.0,FILA,5,후기 사진은 크게 확인 부탁드립니다 남자친구랑 사귀고 첫 커플 신발인데 뭐를 살까 ...,사진 크다 확인 부탁 드리다 남자 친구 사귀다 커플 신발 고민 휠라 에샤페 초코 너...
1,140502.0,FILA,5,엑실러스3 유저의 스피드서브 2 0 기변 후기 테니스 구력 2년 차 그동안 저의 베...,엑실러스3 유저 스피드 서브 테니스 그동안 베이스 라인 든든 지키다 엑실러스 스피드...
2,143962.0,FILA,5,첫 커플신발로 휠라 하레핀1998을 선택했어요 일반적인 스니커즈 디자인이 질려서 고...,커플 신발 휠라 하레핀 스니커즈 디자인 질리다 고민 캐주얼 등산화 느낌 하레핀 마음...


## 3. 100건 벤치마크 (실측 시간 산정)

**목적**: M4 Pro에서 v9 모델의 실제 단건 시간 측정 → 17,014건 총 시간 환산

**유의**:
- 벤치마크 셀 실행 중 다른 GPU 무거운 작업(영상 편집 등) 동시 실행 시 측정값 왜곡됨
- workers=2 기준. 4로 늘릴 경우 별도 셀 추가 후 재측정

In [6]:
# 벤치마크용 100건 무작위 추출 (재현 위해 seed 고정)
BENCH_N = 100
bench_df = fila_complement.sample(BENCH_N, random_state=42).reset_index(drop=True)
print(f'벤치마크 대상: {len(bench_df)}건')

# 실행
t0 = time.time()
bench_result = predict_dataframe(
    bench_df,
    few_shots=None,            # 내부에서 골든셋으로 자동 구성
    content_col='content_clean',
    tokens_col='tokens',
    show_progress=True,
    checkpoint_path=None,      # 벤치마크는 체크포인트 사용 안 함
    golden_path=str(GOLDEN_XLSX),
    workers=2,
)
elapsed_bench = time.time() - t0

per_review = elapsed_bench / BENCH_N
total_eta_h = per_review * len(fila_complement) / 3600

print(f'\n=== 벤치마크 결과 ===')
print(f'  100건 처리:          {elapsed_bench:.1f}초')
print(f'  단건 평균:           {per_review:.2f}초/건')
print(f'  17,014건 추정 시간:  {total_eta_h:.1f}시간 ({total_eta_h/24:.1f}일)')

# 결과 저장
bench_result.to_csv(BENCHMARK_CSV, index=False)
print(f'\n저장: {BENCHMARK_CSV.name}')

벤치마크 대상: 100건
[v5] tokens 컬럼 없음 → KiWi 토크나이저로 자동 생성 중...
  완료: 1000건


추론(v9-핏규칙강화): 100%|██████████| 100/100 [01:44<00:00,  1.05s/it]


=== 벤치마크 결과 ===
  100건 처리:          105.1초
  단건 평균:           1.05초/건
  17,014건 추정 시간:  5.0시간 (0.2일)

저장: absa_fila_benchmark_100.csv


In [7]:
# 벤치마크 라벨 분포 확인 (정상 동작 검증)
for asp in ASPECTS:
    if asp in bench_result.columns:
        dist = bench_result[asp].value_counts(normalize=True).round(3).to_dict()
        print(f'  {asp:<14} {dist}')

# 모든 X로 떨어지면 비정상 (LLM 응답 파싱 실패 가능성)
x_only_aspects = [a for a in ASPECTS if a in bench_result.columns and (bench_result[a] == 'X').all()]
if x_only_aspects:
    print(f'\n⚠ 경고 — 100% X로 예측된 속성: {x_only_aspects}')
    print('  본 추론 진입 전 absa_v9.call_exaone_batch 응답 점검 필요')
else:
    print('\n분포 정상 — 본 추론 진입 가능')

  핏/사이즈          {'P': 0.77, 'X': 0.23}
  소재/내구성         {'X': 0.9, 'P': 0.1}
  기능성            {'X': 0.59, 'P': 0.37, 'N': 0.04}
  디자인            {'P': 0.66, 'X': 0.33, 'N': 0.01}
  브랜드/헤리티지       {'X': 0.77, 'P': 0.22, 'N': 0.01}
  가격/가치          {'X': 0.98, 'P': 0.02}

분포 정상 — 본 추론 진입 가능


## 4. 본 추론 (17,014건) — 체크포인트 기반

**실행 전 확인**:
- 위 벤치마크에서 추정 시간 합리적인지 (10~20시간 권장)
- 라벨 분포가 6속성 모두 P/N/X 섞여 있어야 정상
- Mac sleep 방지 켜기 (터미널에서 `caffeinate -i` 실행 또는 시스템 설정)

**중단 시**:
- Ctrl+C로 중단해도 50건마다 체크포인트가 `absa_fila_complement_checkpoint.csv`에 저장됨
- 해당 셀 재실행 시 자동 이어받기

In [8]:
# ⚠ 본 추론 — 14~24시간 소요 예정. 시작 전 벤치마크 결과 검토 후 실행.

WORKERS = 2  # 안정성 우선. M4 Pro 메모리/발열 여유 시 4로 변경 가능

print(f'시작 시각: {time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'대상 건수:  {len(fila_complement):,}')
print(f'workers:   {WORKERS}')
print(f'체크포인트: {CHECKPOINT_CSV.name} (50건마다)')
print('-' * 60)

t0 = time.time()
predictions = predict_dataframe(
    fila_complement,
    few_shots=None,
    content_col='content_clean',
    tokens_col='tokens',
    show_progress=True,
    checkpoint_path=str(CHECKPOINT_CSV),
    checkpoint_every=50,
    golden_path=str(GOLDEN_XLSX),
    workers=WORKERS,
)
elapsed = time.time() - t0

print('-' * 60)
print(f'완료 시각: {time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'총 소요:   {elapsed/3600:.2f}시간')
print(f'단건 평균: {elapsed/len(fila_complement):.2f}초')

시작 시각: 2026-05-09 14:15:15
대상 건수:  17,014
workers:   2
체크포인트: absa_fila_complement_checkpoint.csv (50건마다)
------------------------------------------------------------
[v5] tokens 컬럼 없음 → KiWi 토크나이저로 자동 생성 중...
  완료: 1000건


추론(v9-핏규칙강화): 100%|██████████| 17014/17014 [4:45:31<00:00,  1.01s/it]  

------------------------------------------------------------
완료 시각: 2026-05-09 19:00:47
총 소요:   4.76시간
단건 평균: 1.01초


## 5. 결과 저장 + 검증

In [9]:
# 최종 parquet 저장 (대시보드/통합 분석용)
predictions.to_parquet(PREDICTIONS_PARQUET, index=False)
print(f'저장: {PREDICTIONS_PARQUET}')
print(f'  행 {len(predictions):,} × 열 {len(predictions.columns)}')

# 6속성 라벨 분포
print('\n=== 속성별 라벨 분포 ===')
for asp in ASPECTS:
    if asp in predictions.columns:
        vc = predictions[asp].value_counts(normalize=True).round(3)
        print(f'  {asp:<14} P={vc.get("P", 0):.3f}  N={vc.get("N", 0):.3f}  X={vc.get("X", 0):.3f}')

# 통합 검증: 라벨러1 sample 과 review_id 중복 0
overlap = set(predictions['review_id']) & fila_sample_ids
print(f'\n라벨러1 sample 과 중복 검증: {len(overlap)}건 (0이어야 정상)')
assert len(overlap) == 0

저장: /Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/absa_exaone/absa_fila_complement_predictions.parquet
  행 17,014 × 열 41

=== 속성별 라벨 분포 ===
  핏/사이즈          P=0.736  N=0.021  X=0.244
  소재/내구성         P=0.127  N=0.025  X=0.848
  기능성            P=0.347  N=0.022  X=0.631
  디자인            P=0.653  N=0.006  X=0.342
  브랜드/헤리티지       P=0.213  N=0.003  X=0.784
  가격/가치          P=0.045  N=0.001  X=0.955

라벨러1 sample 과 중복 검증: 0건 (0이어야 정상)


In [10]:
# 라벨러1 sample 결과(나중에) + 송원우 차집합 결과 통합 미리보기 시뮬레이션
# (실제 통합은 라벨러1의 absa_phase_e_predictions.parquet 도착 후 진행)
print('=== FILA 전수 분석 구성 (예상) ===')
print(f'  라벨러1 stratified FILA: {len(fila_sample_ids):,}건')
print(f'  송원우  차집합 FILA:    {len(predictions):,}건')
print(f'  합계 (FILA 전수):       {len(fila_sample_ids) + len(predictions):,}건')
print(f'  전체 FILA 데이터 행:    {len(fila_all):,}건  (일치해야 정상)')

# 강·약점 Top 3 (P_ratio, N_ratio 기준)
print('\n=== FILA 차집합 강·약점 ===')
rows = []
for asp in ASPECTS:
    if asp in predictions.columns:
        n = len(predictions)
        p_ratio = (predictions[asp] == 'P').sum() / n
        n_ratio = (predictions[asp] == 'N').sum() / n
        rows.append({'속성': asp, 'P_ratio': p_ratio, 'N_ratio': n_ratio,
                     '균형': p_ratio - n_ratio})
summary = pd.DataFrame(rows).sort_values('균형', ascending=False)
print(summary.to_string(index=False, float_format='%.3f'))

=== FILA 전수 분석 구성 (예상) ===
  라벨러1 stratified FILA: 3,014건
  송원우  차집합 FILA:    17,014건
  합계 (FILA 전수):       20,028건
  전체 FILA 데이터 행:    20,028건  (일치해야 정상)

=== FILA 차집합 강·약점 ===
      속성  P_ratio  N_ratio    균형
   핏/사이즈    0.736    0.021 0.715
     디자인    0.653    0.006 0.647
     기능성    0.347    0.022 0.325
브랜드/헤리티지    0.213    0.003 0.210
  소재/내구성    0.127    0.025 0.102
   가격/가치    0.045    0.001 0.044
